In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.crs import CRS
from rasterio.mask import mask, mask as rio_mask
from rasterio.features import geometry_mask, rasterize as rio_rasterize
from rasterio.transform import from_bounds
from rasterio.warp import calculate_default_transform, reproject, Resampling
from scipy.ndimage import uniform_filter, distance_transform_edt, maximum_filter
from scipy.spatial import cKDTree
from shapely.geometry import mapping
from shapely.ops import unary_union
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score, confusion_matrix,
    ConfusionMatrixDisplay, cohen_kappa_score, f1_score,
)
from imblearn.over_sampling import SMOTE
from statsmodels.stats.outliers_influence import variance_inflation_factor
from xgboost import XGBClassifier
import joblib

BASE_DIR = Path(r"D:\IUB\Thesis\GIS")

plt.rcParams.update({"font.family": "Times New Roman", "font.size": 12})


In [ ]:
DEM_PATH = str(BASE_DIR / 'DEM_V2/DEM_V2.tif')
SHAPEFILE = str(BASE_DIR / 'Study Area/SKBJNG.shp')
OUT_DIR = str(BASE_DIR / 'Python')
TARGET_EPSG = 32645
os.makedirs(OUT_DIR, exist_ok=True)
print(' Imports successful. Output directory ready.')


In [ ]:
aoi = gpd.read_file(SHAPEFILE)
print(f'AOI original CRS : {aoi.crs}')
with rasterio.open(DEM_PATH) as src:
    dem_crs = src.crs
    src_dtype = src.dtypes[0]
    src_nodata = src.nodata
    print(f'DEM CRS          : {dem_crs}')
    print(f'DEM dtype        : {src_dtype}')
    print(f'DEM nodata value : {src_nodata}')
    print(f'DEM resolution   : {src.res}')
    print(f'DEM shape        : {src.shape}')
    aoi_reproj = aoi.to_crs(dem_crs)
    shapes = [mapping(geom) for geom in aoi_reproj.geometry]
    clipped_data, clipped_transform = mask(src, shapes, crop=True, nodata=-9999)
    clipped_meta = src.meta.copy()
clipped_data = clipped_data.astype(np.float32)
clipped_data[clipped_data == -9999] = np.nan
if src_nodata is not None and src_nodata != -9999:
    clipped_data[clipped_data == float(src_nodata)] = np.nan
target_crs = CRS.from_epsg(TARGET_EPSG)
height, width = (clipped_data.shape[1], clipped_data.shape[2])
transform_utm, width_utm, height_utm = calculate_default_transform(dem_crs, target_crs, width, height, *rasterio.transform.array_bounds(height, width, clipped_transform))
dem_utm = np.full((1, height_utm, width_utm), np.nan, dtype=np.float32)
reproject(source=clipped_data, destination=dem_utm, src_transform=clipped_transform, src_crs=dem_crs, dst_transform=transform_utm, dst_crs=target_crs, resampling=Resampling.bilinear, src_nodata=np.nan, dst_nodata=np.nan)
dem_arr = dem_utm[0]
res_x = transform_utm.a
res_y = abs(transform_utm.e)
print(f'\nReprojected DEM shape    : {dem_arr.shape}')
print(f'Pixel resolution (m)     : {res_x:.2f} x {res_y:.2f}')
print(f'Elevation range (m)      : {np.nanmin(dem_arr):.2f}  to  {np.nanmax(dem_arr):.2f}')
dem_out_path = os.path.join(OUT_DIR, 'DEM_clipped_UTM45N.tif')
utm_meta = {'driver': 'GTiff', 'dtype': 'float32', 'width': width_utm, 'height': height_utm, 'count': 1, 'crs': target_crs, 'transform': transform_utm, 'nodata': np.nan, 'compress': 'lzw'}
with rasterio.open(dem_out_path, 'w', **utm_meta) as dst:
    dst.write(dem_arr, 1)
print(f'\n Clipped & reprojected DEM saved:\n   {dem_out_path}')


In [ ]:
def save_geotiff(array, filename, meta_template, nodata_val=np.nan):
    out_meta = meta_template.copy()
    out_meta.update({'count': 1, 'dtype': 'float32', 'nodata': nodata_val, 'compress': 'lzw'})
    out_path = os.path.join(OUT_DIR, filename)
    with rasterio.open(out_path, 'w', **out_meta) as dst:
        dst.write(array.astype(np.float32), 1)
    print(f'   Saved : {out_path}')
    return out_path
print(' Helper function defined.')


In [ ]:
def compute_slope_aspect(dem, res_x, res_y):
    z = dem.copy()
    zp = np.pad(z, 1, mode='edge')
    a = zp[:-2, :-2]
    b = zp[:-2, 1:-1]
    c = zp[:-2, 2:]
    d = zp[1:-1, :-2]
    f = zp[1:-1, 2:]
    g = zp[2:, :-2]
    h = zp[2:, 1:-1]
    i = zp[2:, 2:]
    dzdx = (c + 2 * f + i - (a + 2 * d + g)) / (8.0 * res_x)
    dzdy = (g + 2 * h + i - (a + 2 * b + c)) / (8.0 * res_y)
    slope = np.arctan(np.sqrt(dzdx ** 2 + dzdy ** 2))
    aspect = np.arctan2(dzdy, -dzdx)
    nodata_mask = np.isnan(dem)
    slope[nodata_mask] = np.nan
    aspect[nodata_mask] = np.nan
    return (slope, aspect)
slope_rad, aspect_rad = compute_slope_aspect(dem_arr, res_x, res_y)
print(f'Slope range (degrees): {np.nanmin(np.degrees(slope_rad)):.4f}  to  {np.nanmax(np.degrees(slope_rad)):.4f}')
print(' Slope and aspect computed.')


In [ ]:
def compute_tri(dem):
    zp = np.pad(dem.astype(np.float64), 1, mode='edge')
    center = zp[1:-1, 1:-1]
    tri = np.zeros(center.shape, dtype=np.float64)
    offsets = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
    for di, dj in offsets:
        i0 = 1 + di
        i1 = zp.shape[0] - 1 + di
        j0 = 1 + dj
        j1 = zp.shape[1] - 1 + dj
        neighbour = zp[i0:i1, j0:j1]
        tri += (center - neighbour) ** 2
    tri = np.sqrt(tri).astype(np.float32)
    tri[np.isnan(dem)] = np.nan
    return tri
tri = compute_tri(dem_arr)
print(f'TRI range: {np.nanmin(tri):.4f}  to  {np.nanmax(tri):.4f} m')
save_geotiff(tri, 'TRI.tif', utm_meta)
print(' TRI complete.')


In [ ]:
def compute_d8_flow_accumulation(dem, res_x, res_y):
    rows, cols = dem.shape
    row_off = [0, 1, 1, 1, 0, -1, -1, -1]
    col_off = [1, 1, 0, -1, -1, -1, 0, 1]
    dist = [res_x, np.sqrt(res_x ** 2 + res_y ** 2), res_y, np.sqrt(res_x ** 2 + res_y ** 2), res_x, np.sqrt(res_x ** 2 + res_y ** 2), res_y, np.sqrt(res_x ** 2 + res_y ** 2)]
    dem_filled = dem.copy()
    nan_mask = np.isnan(dem_filled)
    dem_filled[nan_mask] = np.nanmean(dem_filled)
    flow_dir = np.full((rows, cols), -1, dtype=np.int8)
    max_drop = np.full((rows, cols), -np.inf)
    for k in range(8):
        r = row_off[k]
        c = col_off[k]
        r0s = max(0, -r)
        r0e = rows - max(0, r)
        c0s = max(0, -c)
        c0e = cols - max(0, c)
        r1s = max(0, r)
        r1e = rows - max(0, -r)
        c1s = max(0, c)
        c1e = cols - max(0, -c)
        drop = (dem_filled[r0s:r0e, c0s:c0e] - dem_filled[r1s:r1e, c1s:c1e]) / dist[k]
        region = (slice(r0s, r0e), slice(c0s, c0e))
        update = drop > max_drop[region]
        max_drop[region][update] = drop[update]
        flow_dir[region][update] = k
    accum = np.ones((rows, cols), dtype=np.float64)
    flat_indices = np.argsort(dem_filled.ravel())[::-1]
    row_idx, col_idx = np.unravel_index(flat_indices, (rows, cols))
    for r, c in zip(row_idx, col_idx):
        if nan_mask[r, c]:
            continue
        k = flow_dir[r, c]
        if k == -1:
            continue
        nr = r + row_off[k]
        nc = c + col_off[k]
        if 0 <= nr < rows and 0 <= nc < cols and (not nan_mask[nr, nc]):
            accum[nr, nc] += accum[r, c]
    accum[nan_mask] = np.nan
    return accum

def compute_twi(dem, slope_rad, res_x, res_y):
    flow_accum = compute_d8_flow_accumulation(dem, res_x, res_y)
    cell_area = res_x * res_y
    As = flow_accum * cell_area / res_x
    tan_slope = np.tan(np.where(slope_rad < 0.001, 0.001, slope_rad))
    twi = np.log(As / tan_slope)
    twi[np.isnan(dem)] = np.nan
    return twi.astype(np.float32)
print('Computing D8 flow accumulation (this may take 1 to 3 minutes on large DEMs)...')
twi = compute_twi(dem_arr, slope_rad, res_x, res_y)
print(f'TWI range: {np.nanmin(twi):.4f}  to  {np.nanmax(twi):.4f}')
save_geotiff(twi, 'TWI.tif', utm_meta)
print(' TWI complete.')


In [ ]:
def compute_tpi(dem, res_x, neighbourhood_radius_m=300):
    half_win = max(1, int(round(neighbourhood_radius_m / res_x)))
    win_size = 2 * half_win + 1
    print(f'   TPI neighbourhood: radius={neighbourhood_radius_m} m, window={win_size}×{win_size} pixels')
    dem_filled = dem.copy()
    nan_mask = np.isnan(dem_filled)
    dem_filled[nan_mask] = np.nanmean(dem_filled)
    focal_mean = uniform_filter(dem_filled, size=win_size, mode='nearest')
    tpi = (dem_filled - focal_mean).astype(np.float32)
    tpi[nan_mask] = np.nan
    return tpi
tpi = compute_tpi(dem_arr, res_x, neighbourhood_radius_m=300)
print(f'TPI range: {np.nanmin(tpi):.4f}  to  {np.nanmax(tpi):.4f} m')
save_geotiff(tpi, 'TPI.tif', utm_meta)
print(' TPI complete.')


In [ ]:
RIVERS_PATH = str(BASE_DIR / 'LULC/Rivers_SKBJNG.shp')
FINAL_AOI = str(BASE_DIR / 'Study Area/SKBJNG_Final.shp')
TARGET_EPSG = 32645
OUT_DIR = str(BASE_DIR / 'Python')
with rasterio.open(os.path.join(OUT_DIR, 'DEM_clipped_UTM45N.tif')) as _src:
    cell_size = _src.res[0]
    print(f'Cell size from DEM : {cell_size:.2f} m')
rivers = gpd.read_file(RIVERS_PATH)
aoi = gpd.read_file(FINAL_AOI)
rivers_utm = rivers.to_crs(epsg=TARGET_EPSG)
aoi_utm = aoi.to_crs(epsg=TARGET_EPSG)
print(f'Rivers  : {len(rivers_utm)} polygons, CRS: {rivers_utm.crs}')
print(f'AOI     : {len(aoi_utm)} features,  CRS: {aoi_utm.crs}')
rivers_clipped = gpd.clip(rivers_utm, aoi_utm)
print(f'Rivers after clip to AOI: {len(rivers_clipped)} features')
bounds = aoi_utm.total_bounds
width_r = int(np.ceil((bounds[2] - bounds[0]) / cell_size))
height_r = int(np.ceil((bounds[3] - bounds[1]) / cell_size))
transform_r = from_bounds(bounds[0], bounds[1], bounds[2], bounds[3], width_r, height_r)
print(f'Output raster : {height_r} rows × {width_r} cols at {cell_size:.1f} m')
print('Rasterising river polygons...')
river_poly_raster = rio_rasterize([(mapping(geom), 1) for geom in rivers_clipped.geometry if geom is not None], out_shape=(height_r, width_r), transform=transform_r, fill=0, dtype=np.uint8, all_touched=False)
print(f'River pixels (polygon fill): {river_poly_raster.sum():,}')
dist = distance_transform_edt(river_poly_raster)
local_max = maximum_filter(dist, size=3)
skeleton = ((dist == local_max) & (river_poly_raster == 1)).astype(np.float32)
print(f'Skeleton (centreline) pixels: {int(skeleton.sum()):,}')
window_radius_m = 5000
half_win = max(1, int(round(window_radius_m / cell_size)))
win_size = 2 * half_win + 1
print(f'DD window : {window_radius_m} m radius : {win_size}×{win_size} pixels')
channel_length_m = skeleton * cell_size
total_channel_len_m = uniform_filter(channel_length_m, size=win_size, mode='constant', cval=0) * win_size ** 2
window_area_km2 = (win_size * cell_size) ** 2 / 1000000.0
dd_final = total_channel_len_m / 1000.0 / window_area_km2
dd_final = dd_final.astype(np.float32)
aoi_mask = geometry_mask([mapping(geom) for geom in aoi_utm.geometry], out_shape=(height_r, width_r), transform=transform_r, invert=False)
dd_final[aoi_mask] = np.nan
print(f'\nDD range : {np.nanmin(dd_final):.4f}  to  {np.nanmax(dd_final):.4f} km/km²')
print(f'DD median: {np.nanmedian(dd_final):.4f} km/km²')
dd_meta = {'driver': 'GTiff', 'dtype': 'float32', 'width': width_r, 'height': height_r, 'count': 1, 'crs': CRS.from_epsg(TARGET_EPSG), 'transform': transform_r, 'nodata': np.nan, 'compress': 'lzw'}
dd_out_path = os.path.join(OUT_DIR, 'Drainage_Density_Final.tif')
with rasterio.open(dd_out_path, 'w', **dd_meta) as dst:
    dst.write(dd_final, 1)
print(f' Drainage Density saved : {dd_out_path}')
skel_path = os.path.join(OUT_DIR, 'River_Centrelines.tif')
skel_meta = dd_meta.copy()
skel_meta['dtype'] = 'uint8'
skel_meta['nodata'] = 255
with rasterio.open(skel_path, 'w', **skel_meta) as dst:
    dst.write(skeleton.astype(np.uint8), 1)
print(f' River centrelines saved : {skel_path}')
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
axes[0].imshow(river_poly_raster, cmap='Blues', interpolation='nearest')
axes[0].set_title('River Polygons (rasterised)', fontsize=11)
axes[0].axis('off')
axes[1].imshow(skeleton, cmap='Blues', interpolation='nearest')
axes[1].set_title('River Centrelines (skeleton)', fontsize=11)
axes[1].axis('off')
vmin, vmax = (np.nanpercentile(dd_final, 2), np.nanpercentile(dd_final, 98))
im = axes[2].imshow(dd_final, cmap='YlOrRd', interpolation='nearest', vmin=vmin, vmax=vmax)
axes[2].set_title('Drainage Density (km/km²)', fontsize=11)
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
plt.tight_layout()
qc_path = os.path.join(OUT_DIR, 'DD_Final_QC.png')
plt.savefig(qc_path, dpi=150, bbox_inches='tight')
plt.show()
print(f' QC plot saved : {qc_path}')


In [ ]:
with rasterio.open(os.path.join(OUT_DIR, 'Drainage_Density_Final.tif')) as _src:
    dd_final = _src.read(1)
    dd_final[dd_final == _src.nodata] = np.nan
layers = [(dem_arr, 'Digital Elevation Model (DEM) (m)', 'terrain', False), (tri, 'Terrain Ruggedness Index (TRI) (m)', 'YlOrRd', False), (twi, 'Topographic Wetness Index (TWI)', 'Blues', False), (tpi, 'Topographic Position Index (TPI) (m)', 'RdBu', True), (dd_final, 'Drainage Density (DD) (km/km²)', 'YlOrRd', False)]
fig, axes = plt.subplots(2, 3, figsize=(20, 13))
axes = axes.flatten()
for idx, (arr, title, cmap, symmetric) in enumerate(layers):
    ax = axes[idx]
    if symmetric:
        abs_max = np.nanpercentile(np.abs(arr), 98)
        vmin, vmax = (-abs_max, abs_max)
    else:
        vmin = np.nanpercentile(arr, 2)
        vmax = np.nanpercentile(arr, 98)
    im = ax.imshow(arr, cmap=cmap, interpolation='nearest', vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=10, fontweight='bold', pad=8)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
axes[5].set_visible(False)
plt.subplots_adjust(wspace=0.15, hspace=0.25)
plot_path = os.path.join(OUT_DIR, 'terrain_parameters_FINAL.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f' Final QC plot saved : {plot_path}')


In [ ]:
LULC_IN = str(BASE_DIR / 'LULC/LULC_SKBJNG.shp')
LULC_OUT = str(BASE_DIR / 'LULC/LULC_SKBJNG_Reclassified.shp')
reclassify_map = {'Waterbody': 'Waterbodies', 'Administrative Area': 'Built Area', 'Commercial Area': 'Built Area', 'Educational Institution': 'Built Area', 'Industrial Area': 'Built Area', 'Monument': 'Built Area', 'Religious Place': 'Built Area', 'Road': 'Built Area', 'Settlement': 'Built Area', 'Terminal': 'Built Area', 'Agricultural Land': 'Vegetation', 'Miscellaneous Landuse': 'Vegetation', 'Recreational Place': 'Bareland', 'Port': 'Bareland', 'Forest': 'Sundarbans'}
lulc = gpd.read_file(LULC_IN)
print(f'Total features     : {len(lulc)}')
print(f'Columns            : {list(lulc.columns)}')
print(f"\nUnique maj_class values ({lulc['maj_class'].nunique()}):")
for cls in sorted(lulc['maj_class'].unique()):
    print(f"   '{cls}'")
lulc['LULC_Class'] = lulc['maj_class'].map(reclassify_map)
unmapped = lulc[lulc['LULC_Class'].isna()]['maj_class'].unique()
if len(unmapped) > 0:
    print(f"\n  WARNING : {len(unmapped)} class(es) in 'maj_class' not found in reclassify_map:")
    for u in unmapped:
        count = (lulc['maj_class'] == u).sum()
        print(f"   '{u}' : {count} features : assigned 'Unclassified'")
    lulc['LULC_Class'] = lulc['LULC_Class'].fillna('Unclassified')
else:
    print('\n All maj_class values successfully mapped.')
print('\nReclassified LULC_Class distribution:')
summary = lulc.groupby('LULC_Class').agg(Features=('LULC_Class', 'count'), Area_km2=('geometry', lambda x: x.to_crs(epsg=32645).area.sum() / 1000000.0)).round(3)
print(summary.to_string())
lulc.to_file(LULC_OUT)
print(f'\n Reclassified shapefile saved : {LULC_OUT}')


In [ ]:
RAINFALL_SHP = str(BASE_DIR / 'Rainfall CRU/Rainfall.shp')
AOI_SHP = str(BASE_DIR / 'Study Area/SKBJNG.shp')
OUT_DIR = str(BASE_DIR / 'Python')
TARGET_EPSG = 32645
with rasterio.open(os.path.join(OUT_DIR, 'DEM_clipped_UTM45N.tif')) as _src:
    cell_size = _src.res[0]
print(f'Cell size from DEM : {cell_size:.2f} m')
rainfall = gpd.read_file(RAINFALL_SHP)
aoi = gpd.read_file(AOI_SHP)
print(f'Rainfall records   : {len(rainfall)}')
print(f"Year range         : {rainfall['Year'].min()}  to  {rainfall['Year'].max()}")
print(f"Unique stations    : {len(rainfall.groupby(['Lat', 'Lon']))}")
station_means = rainfall.groupby(['Lon', 'Lat']).agg(Dry_Mean=('Dry_Season', 'mean'), Wet_Mean=('Wet_Season', 'mean')).reset_index()
print(f'\nStation count after aggregation : {len(station_means)}')
print(f"Dry season mean range (mm)      : {station_means['Dry_Mean'].min():.1f}  to  {station_means['Dry_Mean'].max():.1f}")
print(f"Wet season mean range (mm)      : {station_means['Wet_Mean'].min():.1f}  to  {station_means['Wet_Mean'].max():.1f}")
stations_gdf = gpd.GeoDataFrame(station_means, geometry=gpd.points_from_xy(station_means['Lon'], station_means['Lat']), crs='EPSG:4326')
stations_utm = stations_gdf.to_crs(epsg=TARGET_EPSG)
aoi_utm = aoi.to_crs(epsg=TARGET_EPSG)
bounds = aoi_utm.total_bounds
width_r = int(np.ceil((bounds[2] - bounds[0]) / cell_size))
height_r = int(np.ceil((bounds[3] - bounds[1]) / cell_size))
transform_r = from_bounds(bounds[0], bounds[1], bounds[2], bounds[3], width_r, height_r)
print(f'\nOutput raster : {height_r} rows × {width_r} cols at {cell_size:.1f} m')
cols_idx = np.arange(width_r)
rows_idx = np.arange(height_r)
col_grid, row_grid = np.meshgrid(cols_idx, rows_idx)
px_x = bounds[0] + (col_grid + 0.5) * cell_size
px_y = bounds[3] - (row_grid + 0.5) * cell_size
pixel_coords = np.column_stack([px_x.ravel(), px_y.ravel()])
station_coords = np.column_stack([stations_utm.geometry.x.values, stations_utm.geometry.y.values])

def idw_interpolate(station_coords, values, pixel_coords, power=2, k=None):
    tree = cKDTree(station_coords)
    if k is None:
        k = len(values)
    k = min(k, len(values))
    distances, indices = tree.query(pixel_coords, k=k)
    distances = np.where(distances == 0, 1e-10, distances)
    weights = 1.0 / distances ** power
    weight_sum = weights.sum(axis=1, keepdims=True)
    norm_weights = weights / weight_sum
    interpolated = (norm_weights * values[indices]).sum(axis=1)
    return interpolated.reshape(px_x.shape)
print('\nInterpolating Dry Season rainfall (IDW, power=2)...')
dry_values = stations_utm['Dry_Mean'].values
dry_grid = idw_interpolate(station_coords, dry_values, pixel_coords, power=2)
print('Interpolating Wet Season rainfall (IDW, power=2)...')
wet_values = stations_utm['Wet_Mean'].values
wet_grid = idw_interpolate(station_coords, wet_values, pixel_coords, power=2)
aoi_mask = geometry_mask([mapping(geom) for geom in aoi_utm.geometry], out_shape=(height_r, width_r), transform=transform_r, invert=False)
dry_grid[aoi_mask] = np.nan
wet_grid[aoi_mask] = np.nan
print(f'\nDry Season interpolated range (mm): {np.nanmin(dry_grid):.1f}  to  {np.nanmax(dry_grid):.1f}')
print(f'Wet Season interpolated range (mm): {np.nanmin(wet_grid):.1f}  to  {np.nanmax(wet_grid):.1f}')
rain_meta = {'driver': 'GTiff', 'dtype': 'float32', 'width': width_r, 'height': height_r, 'count': 1, 'crs': CRS.from_epsg(TARGET_EPSG), 'transform': transform_r, 'nodata': np.nan, 'compress': 'lzw'}
dry_path = os.path.join(OUT_DIR, 'Dry_Season_Rainfall.tif')
with rasterio.open(dry_path, 'w', **rain_meta) as dst:
    dst.write(dry_grid.astype(np.float32), 1)
print(f' Dry Season Rainfall saved : {dry_path}')
wet_path = os.path.join(OUT_DIR, 'Wet_Season_Rainfall.tif')
with rasterio.open(wet_path, 'w', **rain_meta) as dst:
    dst.write(wet_grid.astype(np.float32), 1)
print(f' Wet Season Rainfall saved : {wet_path}')
fig, axes = plt.subplots(1, 2, figsize=(14, 8))
vmin_d, vmax_d = (np.nanpercentile(dry_grid, 2), np.nanpercentile(dry_grid, 98))
vmin_w, vmax_w = (np.nanpercentile(wet_grid, 2), np.nanpercentile(wet_grid, 98))
im1 = axes[0].imshow(dry_grid, cmap='YlGnBu', interpolation='nearest', vmin=vmin_d, vmax=vmax_d)
axes[0].scatter([(s - bounds[0]) / cell_size for s in stations_utm.geometry.x], [(bounds[3] - s) / cell_size for s in stations_utm.geometry.y], c='red', s=30, zorder=5, label='CRU stations')
axes[0].set_title('Dry Season Rainfall : Long-term Mean (mm)\n(March to May)', fontsize=11, fontweight='bold')
axes[0].axis('off')
axes[0].legend(loc='lower right', fontsize=9)
plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04, label='mm')
im2 = axes[1].imshow(wet_grid, cmap='YlGnBu', interpolation='nearest', vmin=vmin_w, vmax=vmax_w)
axes[1].scatter([(s - bounds[0]) / cell_size for s in stations_utm.geometry.x], [(bounds[3] - s) / cell_size for s in stations_utm.geometry.y], c='red', s=30, zorder=5, label='CRU stations')
axes[1].set_title('Wet Season Rainfall : Long-term Mean (mm)\n(June to October)', fontsize=11, fontweight='bold')
axes[1].axis('off')
axes[1].legend(loc='lower right', fontsize=9)
plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04, label='mm')
plt.tight_layout()
qc_path = os.path.join(OUT_DIR, 'Rainfall_IDW_QC.png')
plt.savefig(qc_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f' QC plot saved : {qc_path}')


In [ ]:
GW_SHP = str(BASE_DIR / 'Groundwater/Salinity_SKBJNG_plus.shp')
AOI_SHP = str(BASE_DIR / 'Study Area/SKBJNG.shp')
OUT_DIR = str(BASE_DIR / 'Python')
TARGET_EPSG = 32645
PARAMS = [('gwl', 'GwL.tif', 'Groundwater Level (GwL)', 'm', 'Blues'), ('ec_us_cm', 'GwEC.tif', 'Groundwater Electrical Conductivity (GwEC)', 'µS/cm', 'YlOrRd'), ('salinity_p', 'GwS.tif', 'Groundwater Salinity (GwS)', 'ppt', 'RdPu'), ('cl__mg_l', 'GwCl.tif', 'Groundwater Chloride (GwCl)', 'mg/L', 'YlOrBr')]
with rasterio.open(os.path.join(OUT_DIR, 'DEM_clipped_UTM45N.tif')) as _src:
    cell_size = _src.res[0]
print(f'Cell size from DEM : {cell_size:.2f} m')
gw = gpd.read_file(GW_SHP)
aoi = gpd.read_file(AOI_SHP)
print(f'Groundwater points loaded : {len(gw)}')
gw['latitude'] = pd.to_numeric(gw['latitude'], errors='coerce')
gw['longitude'] = pd.to_numeric(gw['longitude'], errors='coerce')
for field, *_ in PARAMS:
    gw[field] = pd.to_numeric(gw[field], errors='coerce')
gw = gw.dropna(subset=['latitude', 'longitude'])
print(f'After dropping missing coords : {len(gw)} points')
print('\nParameter ranges (after cleaning):')
for field, fname, label, unit, _ in PARAMS:
    valid = gw[field].dropna()
    print(f'  {field:<15} n={len(valid):>3}  min={valid.min():.2f}  max={valid.max():.2f}  mean={valid.mean():.2f}  [{unit}]')
gw_gdf = gpd.GeoDataFrame(gw, geometry=gpd.points_from_xy(gw['longitude'], gw['latitude']), crs='EPSG:4326')
gw_utm = gw_gdf.to_crs(epsg=TARGET_EPSG)
aoi_utm = aoi.to_crs(epsg=TARGET_EPSG)
bounds = aoi_utm.total_bounds
width_r = int(np.ceil((bounds[2] - bounds[0]) / cell_size))
height_r = int(np.ceil((bounds[3] - bounds[1]) / cell_size))
transform_r = from_bounds(bounds[0], bounds[1], bounds[2], bounds[3], width_r, height_r)
print(f'\nOutput raster : {height_r} rows × {width_r} cols at {cell_size:.1f} m')
col_grid, row_grid = np.meshgrid(np.arange(width_r), np.arange(height_r))
px_x = bounds[0] + (col_grid + 0.5) * cell_size
px_y = bounds[3] - (row_grid + 0.5) * cell_size
pixel_coords = np.column_stack([px_x.ravel(), px_y.ravel()])
aoi_mask = geometry_mask([mapping(geom) for geom in aoi_utm.geometry], out_shape=(height_r, width_r), transform=transform_r, invert=False)

def idw_interpolate(station_coords, values, pixel_coords, power=2, k=15, batch_size=100000):
    tree = cKDTree(station_coords)
    k = min(k, len(values))
    n_pixels = pixel_coords.shape[0]
    result = np.zeros(n_pixels, dtype=np.float64)
    for start in range(0, n_pixels, batch_size):
        end = min(start + batch_size, n_pixels)
        batch = pixel_coords[start:end]
        dist, idx = tree.query(batch, k=k)
        dist = np.where(dist == 0, 1e-10, dist)
        weights = 1.0 / dist ** power
        norm_w = weights / weights.sum(axis=1, keepdims=True)
        result[start:end] = (norm_w * values[idx]).sum(axis=1)
    return result.reshape(px_x.shape)
raster_meta = {'driver': 'GTiff', 'dtype': 'float32', 'width': width_r, 'height': height_r, 'count': 1, 'crs': CRS.from_epsg(TARGET_EPSG), 'transform': transform_r, 'nodata': np.nan, 'compress': 'lzw'}
grids = {}
for field, fname, label, unit, cmap in PARAMS:
    valid_mask = gw_utm[field].notna()
    pts = np.column_stack([gw_utm.geometry.x.values[valid_mask], gw_utm.geometry.y.values[valid_mask]])
    vals = gw_utm[field].values[valid_mask].astype(np.float64)
    print(f'\nInterpolating {label} using {len(vals)} valid points...')
    grid = idw_interpolate(pts, vals, pixel_coords, power=2)
    grid[aoi_mask] = np.nan
    grids[fname] = grid
    out_path = os.path.join(OUT_DIR, fname)
    with rasterio.open(out_path, 'w', **raster_meta) as dst:
        dst.write(grid.astype(np.float32), 1)
    print(f'    Saved : {out_path}')
    print(f'   Range : {np.nanmin(grid):.2f}  to  {np.nanmax(grid):.2f} {unit}')
fig, axes = plt.subplots(2, 2, figsize=(16, 18))
axes = axes.flatten()
for idx, (field, fname, label, unit, cmap) in enumerate(PARAMS):
    ax = axes[idx]
    grid = grids[fname]
    vmin = np.nanpercentile(grid, 2)
    vmax = np.nanpercentile(grid, 98)
    im = ax.imshow(grid, cmap=cmap, interpolation='nearest', vmin=vmin, vmax=vmax)
    valid_mask = gw_utm[field].notna()
    ax.scatter([(x - bounds[0]) / cell_size for x in gw_utm.geometry.x.values[valid_mask]], [(bounds[3] - y) / cell_size for y in gw_utm.geometry.y.values[valid_mask]], c='black', s=8, zorder=5, alpha=0.6)
    ax.set_title(f'{label} ({unit})', fontsize=11, fontweight='bold', pad=8)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label=unit)
plt.tight_layout()
qc_path = os.path.join(OUT_DIR, 'Groundwater_IDW_QC.png')
plt.savefig(qc_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'\n QC plot saved : {qc_path}')


In [ ]:
SKBJNG_SHP = str(BASE_DIR / 'Study Area/SKBJNG.shp')
SUNDARBAN_SHP = str(BASE_DIR / 'Study Area/Sundarbans.shp')
INPUT_DIR = str(BASE_DIR / 'FINAL LAYERS')
OUTPUT_DIR = str(BASE_DIR / 'Python/FINAL RASTER LAYERS')
FINAL_AOI_OUT = os.path.join(OUTPUT_DIR, 'FINAL_AOI.shp')
TARGET_EPSG = 32645
os.makedirs(OUTPUT_DIR, exist_ok=True)
skbjng = gpd.read_file(SKBJNG_SHP)
sundarbans = gpd.read_file(SUNDARBAN_SHP)
print(f'SKBJNG CRS     : {skbjng.crs}')
print(f'Sundarbans CRS : {sundarbans.crs}')
sundarbans = sundarbans.to_crs(skbjng.crs)
skbjng_union = unary_union(skbjng.geometry)
sundarbans_union = unary_union(sundarbans.geometry)
final_aoi_geom = skbjng_union.difference(sundarbans_union)
print(f'\nSKBJNG geometry type        : {skbjng_union.geom_type}')
print(f'Sundarbans geometry type    : {sundarbans_union.geom_type}')
print(f'FINAL_AOI geometry type     : {final_aoi_geom.geom_type}')
final_aoi_gdf = gpd.GeoDataFrame({'id': [1], 'name': ['FINAL_AOI']}, geometry=[final_aoi_geom], crs=skbjng.crs)
final_aoi_gdf.to_file(FINAL_AOI_OUT)
print(f'\n FINAL_AOI saved : {FINAL_AOI_OUT}')
final_aoi_utm = final_aoi_gdf.to_crs(epsg=TARGET_EPSG)
clip_shapes = [mapping(geom) for geom in final_aoi_utm.geometry]
tif_files = sorted([f for f in os.listdir(INPUT_DIR) if f.lower().endswith('.tif') or f.lower().endswith('.tiff')])
print(f'\nFound {len(tif_files)} GeoTIFF(s) in input folder:')
for f in tif_files:
    print(f'   {f}')
results = []
for fname in tif_files:
    in_path = os.path.join(INPUT_DIR, fname)
    out_path = os.path.join(OUTPUT_DIR, fname)
    try:
        with rasterio.open(in_path) as src:
            clip_gdf = final_aoi_gdf.to_crs(src.crs)
            shapes = [mapping(geom) for geom in clip_gdf.geometry]
            clipped_data, clipped_transform = rio_mask(src, shapes, crop=True, filled=False)
            clipped_data = clipped_data.astype(np.float32).filled(np.nan)
            clipped_data = clipped_data.astype(np.float32)
            if src.nodata is not None and (not np.isnan(src.nodata)):
                clipped_data[clipped_data == src.nodata] = np.nan
            out_meta = src.meta.copy()
            out_meta.update({'driver': 'GTiff', 'dtype': 'float32', 'height': clipped_data.shape[1], 'width': clipped_data.shape[2], 'transform': clipped_transform, 'nodata': np.nan, 'compress': 'lzw'})
        with rasterio.open(out_path, 'w', **out_meta) as dst:
            dst.write(clipped_data)
        size_mb = os.path.getsize(out_path) / 1000000.0
        print(f' {fname:<50} {size_mb:.2f} MB')
        results.append((fname, 'Success', size_mb))
    except Exception as e:
        print(f' {fname:<50} ERROR: {e}')
        results.append((fname, f'ERROR: {e}', 0))
print(f' SUMMARY')
success = [r for r in results if r[1] == 'Success']
failed = [r for r in results if r[1] != 'Success']
print(f'  Total processed : {len(results)}')
print(f'  Successful      : {len(success)}')
print(f'  Failed          : {len(failed)}')
if failed:
    print(f'\n  Failed files:')
    for fname, err, _ in failed:
        print(f'     {fname} : {err}')
print(f'\n  Output directory: {OUTPUT_DIR}')


In [ ]:
PERM_SHP = str(BASE_DIR / 'Soil Data/Soil_Permeability.shp')
SALI_SHP = str(BASE_DIR / 'Soil Data/Soil_Salinity.shp')
AOI_SHP = str(BASE_DIR / 'Python/FINAL RASTER LAYERS/FINAL_AOI.shp')
OUT_DIR = str(BASE_DIR / 'Python/FINAL RASTER LAYERS')
DEM_PATH = str(BASE_DIR / 'Python/DEM_clipped_UTM45N.tif')
TARGET_EPSG = 32645
with rasterio.open(os.path.join(OUT_DIR, 'DEM_clipped_UTM45N.tif')) as _src:
    cell_size = _src.res[0]
print(f'Cell size from DEM : {cell_size:.2f} m')
PERM_MAP = {'Slow': (1, '1 - Very Low'), 'Mostly Slow': (2, '2 - Low'), 'Mostly Slow WS Moderate': (2, '2 - Low'), 'Mostly Slow WS Rapid': (2, '2 - Low'), 'Mixed Slow & Moderate': (3, '3 - Moderate'), 'Mixed Slow & Rapid': (3, '3 - Moderate'), 'Mixed Moderate & Slow': (3, '3 - Moderate'), 'Mostly Moderate': (4, '4 - High'), 'Mostly Moderate WS Slow': (4, '4 - High'), 'Moderate': (5, '5 - Very High')}
SALI_MAP = {'8 - 15 MMHOS/cm': (1, '1 - Very High Salinity'), 'Mostly 8-15 WS 4-8 MMHOS/cm': (1, '1 - Very High Salinity'), '4 - 8 MMHOS/cm': (2, '2 - High Salinity'), 'Mostly 4-8 WS 8-15 MMHOS/cm': (2, '2 - High Salinity'), 'Mostly 4-8 MMHOS/cm': (3, '3 - Moderate Salinity'), '2 - 4 MMHOS/cm': (3, '3 - Moderate Salinity'), 'Mostly <2 WS 4-8 MMHOS/cm': (4, '4 - Low Salinity'), '< 2 MMHOS/cm': (5, '5 - Very Low Salinity')}
aoi = gpd.read_file(AOI_SHP)
aoi_utm = aoi.to_crs(epsg=TARGET_EPSG)
bounds = aoi_utm.total_bounds
width_r = int(np.ceil((bounds[2] - bounds[0]) / cell_size))
height_r = int(np.ceil((bounds[3] - bounds[1]) / cell_size))
transform_r = from_bounds(bounds[0], bounds[1], bounds[2], bounds[3], width_r, height_r)
print(f'Output raster : {height_r} rows × {width_r} cols at {cell_size:.1f} m')
raster_meta = {'driver': 'GTiff', 'dtype': 'float32', 'width': width_r, 'height': height_r, 'count': 1, 'crs': CRS.from_epsg(TARGET_EPSG), 'transform': transform_r, 'nodata': np.nan, 'compress': 'lzw'}
aoi_outside = geometry_mask([mapping(geom) for geom in aoi_utm.geometry], out_shape=(height_r, width_r), transform=transform_r, invert=False)

def process_soil_layer(shp_path, class_field, class_map, layer_name, output_filename):
    print(f' {layer_name}')
    gdf = gpd.read_file(shp_path)
    print(f'  Features loaded : {len(gdf)}')
    gdf['CLASS_NUM'] = gdf[class_field].map(lambda x: class_map.get(str(x).strip(), (0, 'Unclassified'))[0])
    gdf['CLASS_TXT'] = gdf[class_field].map(lambda x: class_map.get(str(x).strip(), (0, 'Unclassified'))[1])
    unclassified = gdf[gdf['CLASS_NUM'] == 0]
    if len(unclassified) > 0:
        print(f'    Unclassified ({len(unclassified)} features):')
        for v in unclassified[class_field].unique():
            print(f'     : "{v}"')
    else:
        print(f'   All features classified')
    print(f'\n  Class distribution:')
    dist = gdf.groupby(['CLASS_NUM', 'CLASS_TXT'], sort=True).size().reset_index(name='n')
    for _, row in dist.iterrows():
        print(f"    {row['CLASS_TXT']:<38} : {row['n']} features")
    gdf_utm = gdf.to_crs(epsg=TARGET_EPSG)
    shapes = [(mapping(row.geometry), int(row['CLASS_NUM'])) for _, row in gdf_utm.iterrows() if row.geometry is not None and row['CLASS_NUM'] > 0]
    raster = rio_rasterize(shapes, out_shape=(height_r, width_r), transform=transform_r, fill=0, dtype=np.uint8, all_touched=False)
    result = raster.astype(np.float32)
    result[aoi_outside] = np.nan
    result[result == 0] = np.nan
    print(f'\n  Value range : {np.nanmin(result):.0f}  to  {np.nanmax(result):.0f}')
    out_path = os.path.join(OUT_DIR, output_filename)
    with rasterio.open(out_path, 'w', **raster_meta) as dst:
        dst.write(result, 1)
    print(f'   Saved : {out_path}')
    return result
perm_raster = process_soil_layer(PERM_SHP, 'SP_CLASS', PERM_MAP, 'Soil Permeability', 'Soil_Permeability.tif')
sali_raster = process_soil_layer(SALI_SHP, 'SI_CLASS', SALI_MAP, 'Soil Salinity', 'Soil_Salinity.tif')
cmap_5 = ListedColormap(['#d73027', '#fc8d59', '#fee090', '#91cf60', '#1a9850'])
norm_5 = BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5, 5.5], cmap_5.N)
colors = ['#d73027', '#fc8d59', '#fee090', '#91cf60', '#1a9850']
fig, axes = plt.subplots(1, 2, figsize=(16, 10))
plots = [(perm_raster, 'Soil Permeability', ['1 - Very Low', '2 - Low', '3 - Moderate', '4 - High', '5 - Very High']), (sali_raster, 'Soil Salinity', ['1 - Very High', '2 - High', '3 - Moderate', '4 - Low', '5 - Very Low'])]
for ax, (raster, title, labels) in zip(axes, plots):
    ax.imshow(raster, cmap=cmap_5, norm=norm_5, interpolation='nearest')
    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
    ax.axis('off')
    ax.legend(handles=[Patch(color=colors[i], label=labels[i]) for i in range(5)], loc='lower left', fontsize=8, title='Classes', title_fontsize=9, framealpha=0.9)
plt.tight_layout()
qc_path = os.path.join(OUT_DIR, 'Soil_Classification_QC.png')
plt.savefig(qc_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'\n QC plot saved : {qc_path}')


In [ ]:
B4_PATH = str(BASE_DIR / 'NDVI/B4_SK.tif')
B5_PATH = str(BASE_DIR / 'NDVI/B5_SK.tif')
FINAL_AOI = str(BASE_DIR / 'Python/FINAL RASTER LAYERS/FINAL_AOI.shp')
OUTPUT_DIR = str(BASE_DIR / 'Python/FINAL RASTER LAYERS')
TARGET_EPSG = 32645

def read_reproject(src_path, target_epsg):
    target_crs = CRS.from_epsg(target_epsg)
    with rasterio.open(src_path) as src:
        print(f'  {os.path.basename(src_path)}')
        print(f'    CRS      : {src.crs}')
        print(f'    Shape    : {src.height} × {src.width}')
        print(f'    dtype    : {src.dtypes[0]}')
        print(f'    nodata   : {src.nodata}')
        transform, width, height = calculate_default_transform(src.crs, target_crs, src.width, src.height, *src.bounds)
        dest = np.full((height, width), np.nan, dtype=np.float32)
        reproject(source=rasterio.band(src, 1), destination=dest, src_transform=src.transform, src_crs=src.crs, dst_transform=transform, dst_crs=target_crs, resampling=Resampling.bilinear, src_nodata=src.nodata, dst_nodata=np.nan)
    meta = {'driver': 'GTiff', 'dtype': 'float32', 'width': width, 'height': height, 'count': 1, 'crs': target_crs, 'transform': transform, 'nodata': np.nan, 'compress': 'lzw'}
    return (dest, meta, transform, width, height)
print('Reading and reprojecting bands...')
b4, meta_b4, transform, width, height = read_reproject(B4_PATH, TARGET_EPSG)
b5, meta_b5, _, _, _ = read_reproject(B5_PATH, TARGET_EPSG)
assert b4.shape == b5.shape, f'Shape mismatch after reproject: B4={b4.shape}, B5={b5.shape}. Check that both rasters cover the same extent.'
print(f'\nBoth bands reprojected: {height} rows × {width} cols ')
if np.nanmax(b4) > 2.0:
    print('Landsat Collection 2 DNs detected : applying scale + offset...')
    b4 = np.clip(b4 * 2.75e-05 + -0.2, 0, 1)
    b5 = np.clip(b5 * 2.75e-05 + -0.2, 0, 1)
    print(f'B4 reflectance range : {np.nanmin(b4):.4f}  to  {np.nanmax(b4):.4f}')
    print(f'B5 reflectance range : {np.nanmin(b5):.4f}  to  {np.nanmax(b5):.4f}')
else:
    print('Bands already in reflectance (0 to 1) : no scaling applied.')
denominator = b5 + b4
ndvi = np.where((denominator == 0) | np.isnan(denominator), np.nan, (b5 - b4) / denominator).astype(np.float32)
ndvi[(ndvi < -1) | (ndvi > 1)] = np.nan
print(f'\nNDVI computed:')
print(f'  Range  : {np.nanmin(ndvi):.4f}  to  {np.nanmax(ndvi):.4f}')
print(f'  Mean   : {np.nanmean(ndvi):.4f}')
print(f'  Valid  : {100 * np.sum(~np.isnan(ndvi)) / ndvi.size:.1f}%')
ndvi_full_path = os.path.join(OUTPUT_DIR, '_NDVI_temp.tif')
with rasterio.open(ndvi_full_path, 'w', **meta_b4) as dst:
    dst.write(ndvi, 1)
aoi = gpd.read_file(FINAL_AOI)
aoi_utm = aoi.to_crs(epsg=TARGET_EPSG)
shapes = [mapping(geom) for geom in aoi_utm.geometry]
print(f'\nClipping to FINAL_AOI...')
with rasterio.open(ndvi_full_path) as src:
    clipped, clipped_transform = rio_mask(src, shapes, crop=True, nodata=np.nan, filled=True)
clipped = clipped.astype(np.float32)
clipped_meta = meta_b4.copy()
clipped_meta.update({'height': clipped.shape[1], 'width': clipped.shape[2], 'transform': clipped_transform})
ndvi_out = os.path.join(OUTPUT_DIR, 'NDVI.tif')
with rasterio.open(ndvi_out, 'w', **clipped_meta) as dst:
    dst.write(clipped)
os.remove(ndvi_full_path)
print(f' NDVI saved : {ndvi_out}')
print(f'   Shape  : {clipped.shape[1]} rows × {clipped.shape[2]} cols')
print(f'   Range  : {np.nanmin(clipped):.4f}  to  {np.nanmax(clipped):.4f}')
ndvi_plot = clipped[0]
vmin = np.nanpercentile(ndvi_plot, 2)
vmax = np.nanpercentile(ndvi_plot, 98)
fig, ax = plt.subplots(figsize=(10, 12))
im = ax.imshow(ndvi_plot, cmap='RdYlGn', interpolation='nearest', vmin=vmin, vmax=vmax)
ax.set_title('NDVI : FINAL_AOI\n(Landsat 8/9, 30m)', fontsize=12, fontweight='bold')
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04, label='NDVI')
plt.tight_layout()
qc_path = os.path.join(OUTPUT_DIR, 'NDVI_QC.png')
plt.savefig(qc_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f' QC plot saved : {qc_path}')


In [ ]:
FINAL_DIR = str(BASE_DIR / 'Python/FINAL RASTER LAYERS')
TARGET_EPSG = 32645
REF_FILE = os.path.join(FINAL_DIR, 'DEM_clipped_UTM45N.tif')
with rasterio.open(REF_FILE) as ref:
    ref_crs = ref.crs
    ref_transform = ref.transform
    ref_width = ref.width
    ref_height = ref.height
    ref_bounds = ref.bounds
print(f'Reference grid : {ref_height} × {ref_width}')
print(f'Reference CRS  : {ref_crs}')
print(f'Reference res  : {ref.res}')
TO_FIX = [('DIS_Rivers.tif', Resampling.bilinear), ('LULC.tif', Resampling.nearest), ('NDVI.tif', Resampling.bilinear)]
for fname, resampling in TO_FIX:
    in_path = os.path.join(FINAL_DIR, fname)
    out_path = os.path.join(FINAL_DIR, fname.replace('.tif', '_aligned.tif'))
    with rasterio.open(in_path) as src:
        print(f'\nProcessing : {fname}')
        print(f'  Input  : {src.height} × {src.width}, CRS={src.crs}, dtype={src.dtypes[0]}')
        out_dtype = np.float32
        nodata_val = np.nan
        dest = np.full((ref_height, ref_width), np.nan, dtype=np.float32)
        reproject(source=rasterio.band(src, 1), destination=dest, src_transform=src.transform, src_crs=src.crs, dst_transform=ref_transform, dst_crs=ref_crs, resampling=resampling, src_nodata=src.nodata, dst_nodata=np.nan)
        out_meta = {'driver': 'GTiff', 'dtype': 'float32', 'width': ref_width, 'height': ref_height, 'count': 1, 'crs': ref_crs, 'transform': ref_transform, 'nodata': np.nan, 'compress': 'lzw'}
    with rasterio.open(out_path, 'w', **out_meta) as dst:
        dst.write(dest, 1)
    print(f'  Output : {ref_height} × {ref_width}, CRS={ref_crs}')
    print(f'  Saved  : {out_path}')
print(f' VERIFICATION : all files should now match reference grid')
all_files = [f for f in os.listdir(FINAL_DIR) if f.endswith('.tif')]
ref_shape = (ref_height, ref_width)
for f in sorted(all_files):
    with rasterio.open(os.path.join(FINAL_DIR, f)) as src:
        shape_ok = src.shape == ref_shape
        crs_ok = src.crs.to_epsg() == TARGET_EPSG
        status = '' if shape_ok and crs_ok else ''
        print(f'  {status} {f:<50} {str(src.shape):>15} EPSG:{src.crs.to_epsg()}')


In [ ]:
FINAL_DIR = str(BASE_DIR / 'Python/FINAL RASTER LAYERS')
REF_FILE = os.path.join(FINAL_DIR, 'DEM_clipped_UTM45N.tif')
with rasterio.open(REF_FILE) as ref:
    ref_crs = ref.crs
    ref_transform = ref.transform
    ref_width = ref.width
    ref_height = ref.height
in_path = os.path.join(FINAL_DIR, 'Drainage_Density_Final.tif')
out_path = os.path.join(FINAL_DIR, 'Drainage_Density_aligned.tif')
with rasterio.open(in_path) as src:
    print(f'Input  : {src.height} × {src.width}')
    dest = np.full((ref_height, ref_width), np.nan, dtype=np.float32)
    reproject(source=rasterio.band(src, 1), destination=dest, src_transform=src.transform, src_crs=src.crs, dst_transform=ref_transform, dst_crs=ref_crs, resampling=Resampling.bilinear, src_nodata=src.nodata, dst_nodata=np.nan)
out_meta = {'driver': 'GTiff', 'dtype': 'float32', 'width': ref_width, 'height': ref_height, 'count': 1, 'crs': ref_crs, 'transform': ref_transform, 'nodata': np.nan, 'compress': 'lzw'}
with rasterio.open(out_path, 'w', **out_meta) as dst:
    dst.write(dest, 1)
print(f'Output : {ref_height} × {ref_width}')
print(f' Saved : {out_path}')


In [ ]:
FINAL_DIR = str(BASE_DIR / "Python/FINAL RASTER LAYERS")
OUT_DIR = str(BASE_DIR / "OUTPUT")
GW_SHP = str(BASE_DIR / "Groundwater/Salinity_SKBJNG_plus.shp")
os.makedirs(OUT_DIR, exist_ok=True)

RASTER_FILES = {
    "DEM": "DEM_clipped_UTM45N.tif",
    "DD": "Drainage_Density_aligned.tif",
    "DIS_Rivers": "DIS_Rivers_aligned.tif",
    "GwCl": "GwCl.tif",
    "GwEC": "GwEC.tif",
    "GwL": "GwL.tif",
    "GwS": "GwS.tif",
    "LULC": "LULC_aligned.tif",
    "NDVI": "NDVI_aligned.tif",
    "Soil_Perm": "Soil_Permeability.tif",
    "Soil_Sali": "Soil_Salinity.tif",
    "TPI": "TPI.tif",
    "TRI": "TRI.tif",
    "TWI": "TWI.tif",
    "Rain_Dry": "Dry_Season_Rainfall.tif",
    "Rain_Wet": "Wet_Season_Rainfall.tif",
}

gw = gpd.read_file(GW_SHP)
longitude = pd.to_numeric(gw["longitude"], errors="coerce")
latitude = pd.to_numeric(gw["latitude"], errors="coerce")
ec_vals = pd.to_numeric(gw["ec_us_cm"], errors="coerce")
valid_rows = longitude.notna() & latitude.notna() & ec_vals.notna()
gw = gw.loc[valid_rows].copy()
gw_utm = gpd.GeoDataFrame(
    gw.drop(columns="geometry"),
    geometry=gpd.points_from_xy(longitude[valid_rows], latitude[valid_rows]),
    crs="EPSG:4326",
).to_crs(epsg=32645)

coords = [(geom.x, geom.y) for geom in gw_utm.geometry]
extracted = {}
for param, filename in RASTER_FILES.items():
    with rasterio.open(os.path.join(FINAL_DIR, filename)) as src:
        values = np.asarray([v[0] for v in src.sample(coords)], dtype=float)
        if src.nodata is not None and not np.isnan(src.nodata):
            values[values == src.nodata] = np.nan
        extracted[param] = values

df = pd.DataFrame(extracted)
df["ec_us_cm"] = ec_vals.loc[valid_rows].to_numpy()
df["label"] = (df["ec_us_cm"] < 1000).astype(int)
df["latitude"] = latitude.loc[valid_rows].to_numpy()
df["longitude"] = longitude.loc[valid_rows].to_numpy()
feature_cols = list(RASTER_FILES)
df_clean = df.replace([np.inf, -np.inf], np.nan).dropna(subset=feature_cols).copy()
print(f"Samples: {len(df)}; complete samples: {len(df_clean)}")
print(df_clean["label"].value_counts().sort_index())


In [ ]:
X_vif = df_clean[feature_cols].copy()
removed_params = []

while True:
    vif_data = pd.DataFrame({
        "Parameter": X_vif.columns,
        "VIF": [variance_inflation_factor(X_vif.values, i)
                for i in range(X_vif.shape[1])],
    })
    if not np.isfinite(vif_data["VIF"]).all():
        raise ValueError("VIF contains nonfinite values. Check the input parameters.")
    max_vif = vif_data["VIF"].max()
    if max_vif <= 15:
        break
    worst_param = vif_data.loc[vif_data["VIF"].idxmax(), "Parameter"]
    removed_params.append({"Parameter": worst_param, "VIF": max_vif})
    X_vif = X_vif.drop(columns=worst_param)

final_params = list(X_vif.columns)
expected_params = ["DD", "DIS_Rivers", "GwCl", "GwEC", "GwL",
                   "GwS", "LULC", "Soil_Perm", "TPI", "TRI"]
if final_params != expected_params:
    raise ValueError(f"VIF selection differs from the 10 study parameters: {final_params}")

print(vif_data.to_string(index=False))
vif_data.to_csv(os.path.join(OUT_DIR, "VIF_Final.csv"), index=False)
pd.DataFrame(removed_params).to_csv(os.path.join(OUT_DIR, "VIF_Removed.csv"), index=False)
df_clean.to_csv(os.path.join(OUT_DIR, "GW_Samples_Extracted.csv"), index=False)
np.save(os.path.join(OUT_DIR, "final_params.npy"), np.asarray(final_params))


In [ ]:
X = df_clean[final_params].to_numpy()
y = df_clean["label"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_sm)
X_test_sc = scaler.transform(X_test)
joblib.dump(scaler, os.path.join(OUT_DIR, "scaler.pkl"))

print(f"Training samples: {len(X_train)}; testing samples: {len(X_test)}")
print("Training classes:", np.unique(y_train, return_counts=True))
print("Testing classes:", np.unique(y_test, return_counts=True))
print("Training classes after SMOTE:", np.unique(y_train_sm, return_counts=True))


In [ ]:
models = {
    "SVM": SVC(kernel="rbf", C=10, gamma="scale", probability=True,
               class_weight="balanced", random_state=42),
    "RF": RandomForestClassifier(n_estimators=200, max_depth=None,
                                 min_samples_leaf=2, class_weight="balanced",
                                 random_state=42, n_jobs=-1),
    "GBM": GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                      max_depth=4, subsample=0.8, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8,
                             eval_metric="logloss", random_state=42, n_jobs=-1),
    "KNN": KNeighborsClassifier(n_neighbors=7, weights="distance",
                                metric="euclidean", n_jobs=-1),
}
scaled_models = {"SVM", "KNN"}
y_preds = {}
y_probs = {}

for name, model in models.items():
    train_values = X_train_sc if name in scaled_models else X_train_sm
    test_values = X_test_sc if name in scaled_models else X_test
    model.fit(train_values, y_train_sm)
    y_preds[name] = model.predict(test_values)
    y_probs[name] = model.predict_proba(test_values)[:, 1]
    joblib.dump(model, os.path.join(OUT_DIR, f"model_{name}.pkl"))
    print(name)
    print(classification_report(y_test, y_preds[name],
                                target_names=["Nonfresh", "Fresh"], zero_division=0))


In [ ]:
rows = []
for name in models:
    cm = confusion_matrix(y_test, y_preds[name], labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_preds[name]),
        "Sensitivity": tp / (tp + fn) if tp + fn else 0,
        "Specificity": tn / (tn + fp) if tn + fp else 0,
        "Precision": tp / (tp + fp) if tp + fp else 0,
        "F1": f1_score(y_test, y_preds[name], zero_division=0),
        "Kappa": cohen_kappa_score(y_test, y_preds[name]),
        "AUC": roc_auc_score(y_test, y_probs[name]),
        "Average precision": average_precision_score(y_test, y_probs[name]),
    })

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(os.path.join(OUT_DIR, "Validation_Metrics.csv"), index=False)
metrics_df[["Model", "Accuracy", "AUC"]].to_csv(
    os.path.join(OUT_DIR, "Model_Performance.csv"), index=False
)
print(metrics_df.round(4).to_string(index=False))


In [ ]:
colors = {"SVM": "#d6604d", "RF": "#2166ac", "GBM": "#1a9850",
          "XGBoost": "#b8860b", "KNN": "#762a83"}

fig, ax = plt.subplots(figsize=(8, 7))
for name in models:
    fpr, tpr, _ = roc_curve(y_test, y_probs[name])
    auc = roc_auc_score(y_test, y_probs[name])
    ax.plot(fpr, tpr, color=colors[name], label=f"{name} (AUC = {auc:.3f})")
ax.plot([0, 1], [0, 1], color="grey", linestyle="dashed", label="Random classifier")
ax.set(xlabel="False positive rate", ylabel="True positive rate",
       title="ROC curves", xlim=(0, 1), ylim=(0, 1.02))
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "Validation_ROC.png"), dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
baseline = y_test.mean()
ax.axhline(baseline, color="grey", linestyle="dashed",
           label=f"Random classifier (AP = {baseline:.3f})")
for name in models:
    precision, recall, _ = precision_recall_curve(y_test, y_probs[name])
    ap = average_precision_score(y_test, y_probs[name])
    ax.plot(recall, precision, color=colors[name], label=f"{name} (AP = {ap:.3f})")
ax.set(xlabel="Recall", ylabel="Precision", title="Precision recall curves",
       xlim=(0, 1), ylim=(0, 1.05))
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "Validation_PR_Curve.png"), dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, name in zip(axes.flat, models):
    cm = confusion_matrix(y_test, y_preds[name], labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=["Nonfresh", "Fresh"]).plot(
        ax=ax, cmap="Blues", colorbar=False
    )
    ax.set_title(name)
axes.flat[5].axis("off")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "Validation_ConfusionMatrices.png"),
            dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(17, 3))
ax.axis("off")
table = ax.table(cellText=metrics_df.round(4).values.tolist(),
                 colLabels=metrics_df.columns, cellLoc="center", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)
for column in range(len(metrics_df.columns)):
    table[(0, column)].set_facecolor("#2166ac")
    table[(0, column)].get_text().set_color("white")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "Validation_MetricsTable.png"),
            dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
reference = os.path.join(FINAL_DIR, "DEM_clipped_UTM45N.tif")
with rasterio.open(reference) as src:
    ref_meta = src.meta.copy()
    ref_shape = src.shape
    ref_transform = src.transform
    ref_crs = src.crs

band_arrays = {}
for param in final_params:
    with rasterio.open(os.path.join(FINAL_DIR, RASTER_FILES[param])) as src:
        if src.shape != ref_shape or src.transform != ref_transform or src.crs != ref_crs:
            raise ValueError(f"Raster grid differs from the reference: {param}")
        band_arrays[param] = src.read(1, masked=True).astype(np.float32).filled(np.nan)

valid_pixels = np.logical_and.reduce([np.isfinite(arr) for arr in band_arrays.values()])
X_all = np.column_stack([band_arrays[param][valid_pixels] for param in final_params])
ref_meta.update(dtype="float32", nodata=np.nan, compress="lzw", count=1)
batch_size = 500_000

for name, model in models.items():
    probabilities = np.empty(len(X_all), dtype=np.float32)
    for start in range(0, len(X_all), batch_size):
        end = min(start + batch_size, len(X_all))
        batch = X_all[start:end]
        if name in scaled_models:
            batch = scaler.transform(batch)
        probabilities[start:end] = model.predict_proba(batch)[:, 1]
    probability_map = np.full(ref_shape, np.nan, dtype=np.float32)
    probability_map[valid_pixels] = probabilities
    filename = os.path.join(OUT_DIR, f"FreshGW_{name}.tif")
    with rasterio.open(filename, "w", **ref_meta) as dst:
        dst.write(probability_map, 1)
    print(filename)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 14))
for ax, name in zip(axes.flat, models):
    with rasterio.open(os.path.join(OUT_DIR, f"FreshGW_{name}.tif")) as src:
        probability_map = src.read(1, masked=True)
        bounds = src.bounds
    image = ax.imshow(probability_map, cmap="RdYlGn", vmin=0, vmax=1,
                      extent=[bounds.left, bounds.right, bounds.bottom, bounds.top])
    ax.set(title=name, xlabel="Easting (m)", ylabel="Northing (m)")
    ax.ticklabel_format(style="sci", axis="both", scilimits=(0, 0))
axes.flat[5].axis("off")
fig.colorbar(image, ax=axes.flat[5], label="Probability of fresh groundwater")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "FreshGW_PotentialZones_5Models.png"),
            dpi=300, bbox_inches="tight")
plt.show()
